# 07: Realtime Monitoring and API Pull

This notebook is dedicated to near real-time benchmark pulls and monitoring checks.

Primary feed in this version:
- FRED Brent spot (`DCOILBRENTEU`)

You can extend this notebook with additional UK retail or government feeds without affecting modeling reproducibility in 06.

## 7.1 Load realtime API data (FRED Brent)

Summary:
- Pulls daily Brent benchmark price in USD/barrel.
- Converts to monthly average for compatibility with the modeling notebooks.

In [1]:
import pandas as pd
from io import StringIO
import ssl
import urllib.request
import plotly.express as px

fred_url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU'

try:
    realtime_brent = pd.read_csv(fred_url)
except Exception:
    insecure_ctx = ssl._create_unverified_context()
    with urllib.request.urlopen(fred_url, context=insecure_ctx) as resp:
        realtime_brent = pd.read_csv(StringIO(resp.read().decode('utf-8')))

realtime_brent.columns = ['date', 'brent_usd_per_barrel']
realtime_brent['date'] = pd.to_datetime(realtime_brent['date'], errors='coerce')
realtime_brent['brent_usd_per_barrel'] = pd.to_numeric(realtime_brent['brent_usd_per_barrel'], errors='coerce')
realtime_brent = realtime_brent.dropna().sort_values('date')

realtime_brent_monthly = realtime_brent.set_index('date')['brent_usd_per_barrel'].resample('MS').mean()

print('Realtime pull succeeded.')
print('Daily range:', realtime_brent['date'].min(), 'to', realtime_brent['date'].max())
print('Latest daily Brent (USD/bbl):', round(realtime_brent['brent_usd_per_barrel'].iloc[-1], 2))
print('Latest monthly Brent mean (USD/bbl):', round(realtime_brent_monthly.iloc[-1], 2))

Realtime pull succeeded.
Daily range: 1987-05-20 00:00:00 to 2026-04-02 00:00:00
Latest daily Brent (USD/bbl): 127.61
Latest monthly Brent mean (USD/bbl): 123.58


## 7.2 Quick monitoring chart

In [2]:
recent_monthly = realtime_brent_monthly.tail(36).reset_index()
recent_monthly.columns = ['date', 'brent_usd_per_barrel']

fig = px.line(
    recent_monthly,
    x='date',
    y='brent_usd_per_barrel',
    title='Brent Spot Monthly Mean (Last 36 Months)',
    labels={'brent_usd_per_barrel': 'USD per barrel', 'date': 'Date'}
)
fig.update_traces(line=dict(width=3))
fig.show()

## 7.3 UK live data guidance

Yes, UK live/near-live data exists, but it is less centralized than Brent benchmarks.

Practical approach:
1. Keep Brent realtime as your global signal.
2. Add UK retail weekly/monthly source in a separate ingestion cell.
3. Convert units/currency before combining into a single monitoring view.